In [2]:
# CodeGrade step0

from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr
import os
import tarfile
import joblib # Import joblib directly
from sklearn.datasets._base import _pkl_filepath, get_data_home
import statsmodels.api as sm
import statsmodels.formula.api as smf


archive_path = "cal_housing.tgz" # change the path if it's not in the current directory
data_home = get_data_home(data_home=None) # change data_home if you are not using ~/scikit_learn_data
if not os.path.exists(data_home):
    os.makedirs(data_home)
filepath = _pkl_filepath(data_home, 'cal_housing.pkz')

with tarfile.open(mode="r:gz", name=archive_path) as f:
    cal_housing = np.loadtxt(
        f.extractfile('CaliforniaHousing/cal_housing.data'),
        delimiter=',')
    # Columns are not in the same order compared to the previous
    # URL resource on lib.stat.cmu.edu
    columns_index = [8, 7, 2, 3, 4, 5, 6, 1, 0]
    cal_housing = cal_housing[:, columns_index]

    joblib.dump(cal_housing, filepath, compress=6) # Now using the directly imported joblib

# Load the dataset
california = fetch_california_housing(as_frame=True)
data = california.data
target = california.target

In [5]:
X = data[['MedInc', 'HouseAge', 'AveRooms']]
y = target
X = sm.add_constant(X)  # Add a constant term for the intercept
baseline_model = smf.ols(formula='y ~ X', data=data).fit()
R_squared = baseline_model.rsquared
round(R_squared, 4)

0.5121

In [6]:
data['MedInc_squared'] = data['MedInc'] ** 2
nonlinear_model = smf.ols(formula='y ~ X + MedInc_squared', data=data).fit()
R_squared_nonlinear = nonlinear_model.rsquared
round(R_squared_nonlinear, 6)

0.52017

In [7]:
interaction_model = smf.ols(formula='y ~ X + MedInc_squared + MedInc*AveRooms', data=data).fit()
R_squared_interaction = interaction_model.rsquared
round(R_squared_interaction, 6)

0.520213

In [11]:
median_income_threshold = data['MedInc'].median()
data['HighIncome'] = np.where(data['MedInc'] > median_income_threshold, 1, 0)
indicator_model = smf.ols(formula='y ~ X + MedInc_squared + HighIncome', data=data).fit()
R_squared_indicator = indicator_model.rsquared    
round(R_squared_indicator, 6)

0.520496

In [9]:
data['log_AveRooms'] = np.log(data['AveRooms'])
interaction_model = smf.ols(formula='y ~ X + MedInc_squared + log_AveRooms', data=data).fit()
R_squared_log_interaction = interaction_model.rsquared
round(R_squared_log_interaction, 6)

0.54855

In [12]:
# CodeGrade step6
log_model = smf.ols(formula='y ~ X + MedInc_squared + log_AveRooms', data=data).fit()
log_model.resid.shape

(20640,)